# Récupérer les abstract via Crossref

In [1]:
#chargement des modules
import pandas as pd
import datetime
import os
import json
from io import StringIO
import numpy as np
import plotly.express as px
import re

In [2]:
import s3fs #pour connecter au bucket

In [3]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})


BUCKET_OUT = "aluneau"



In [4]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl"
print(file_path)
list_id_pub = []
datas =[]
with fs.open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data = json.loads(line)
            # Extract specific fields
            ids = data.get('all_ids')
            list_id_pub.append(ids[0])
            datas.append(data)
            
        except json.JSONDecodeError:
            continue  # Skip invalid lines

aluneau/Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl


La cellule ci-dessous récupèrent les données qui nous intéressent depuis le fichier Json. Ces données seront ensuitre structurées dans un dataframe constitué d'une ligne par publication. Certaines de ces données ne sont en effet pas disponibles dans le fichier "bso-publications-latest_199318270_enriched.csv".


In [7]:
list_row = []


for n, x in enumerate(list_id_pub):
    dict_pub = datas[n]
    dict_row = {"id":x,
                "pub_type": dict_pub["genre"]
               }
    list_row.append(dict_row)
        
        
df0 = pd.DataFrame.from_dict(list_row)

In [8]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.csv"
with fs.open(file_path) as csvin:
    df01 = pd.read_csv(csvin, sep =";")

df1 = df01.merge(df0, on = ["id"], how = "left")

In [10]:
import requests
from requests.exceptions import HTTPError
import urllib.parse
import tqdm

In [16]:
CREF_API_ENDPOINT = "https://api.crossref.org/v1/works"


df1b = df1.loc[~df1.doi.isna()]
df1b

,observation_date,id,doi,pmid,hal_id,year,title,journal_issns,journal_issn_l,journal_name,...,software_created,software_shared,data_used,data_created,data_shared,missing_doi_in_hal,has_doi_in_hal,doi_in_hal,bso_country,pub_type
4,2024Q4,doi10.3917/cliop.028.0029,10.3917/cliop.028.0029,NaN,hal-03819403,2022,Du cas Schreber et de ce qu’on ne doit pas ose...,2100-0670,2100-0670,Cliopsy,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.3917/cliop.028.0029,fr,journal-article
6,2024Q4,doi10.1590/198053142861,10.1590/198053142861,NaN,hal-03325283,2014,A educação infantil e a questão da escola: o c...,0100-1574,0100-1574,Cadernos de Pesquisa,...,False,False,False,False,False,NaN,1.0,10.1590/198053142861,fr|other,journal-article
12,2024Q4,doi10.1109/tsp.2017.8076097,10.1109/tsp.2017.8076097,NaN,hal-03259338,2017,Multispectral palmprint recognition: A state-o...,NaN,NaN,2017 40th International Conference on Telecomm...,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.1109/tsp.2017.8076097,fr,proceedings
19,2024Q4,doi10.4000/hybrid.460,10.4000/hybrid.460,NaN,hal-04173632,2018,Grégory Chatonsky: the art of an archaeofictio...,2276-3538,2276-3538,Hybrid,...,False,False,True,True,False,NaN,1.0,10.4000/hybrid.460,fr|other,journal-article
33,2024Q4,doi10.1016/j.psfr.2014.02.002,10.1016/j.psfr.2014.02.002,NaN,hal-02979729,2014,Apprendre à résoudre des problèmes en favorisa...,0033-2984,0033-2984,Psychologie Française,...,False,False,False,False,False,NaN,1.0,10.1016/j.psfr.2014.02.002,other|fr,journal-article
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45475,2024Q4,doi10.1016/j.compstruct.2015.11.075,10.1016/j.compstruct.2015.11.075,NaN,hal-01912588,2016,Numerical simulation of the nonlinear static b...,0263-8223,0263-8223,Composite Structures,...,True,False,True,False,False,NaN,1.0,10.1016/j.compstruct.2015.11.075,fr|other,journal-article
45486,2024Q4,doi10.3917/sava.056.0039,10.3917/sava.056.0039,NaN,hal-04112191,2021,Les banques centrales dans la pandémie : routi...,1958-7856,1958-5535,Savoir/Agir,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.3917/sava.056.0039,fr,journal-article
45488,2024Q4,doi10.1007/s11856-018-1634-z,10.1007/s11856-018-1634-z,NaN,document,2018,Invariant measures for Cartesian powers of Cha...,"0021-2172,1565-8511",0021-2172,Israel Journal of Mathematics,...,False,False,False,False,False,NaN,1.0,10.1007/s11856-018-1634-z,fr|other,journal-article
45494,2024Q4,doi10.4000/pallas.19740,10.4000/pallas.19740,NaN,halshs-03929927,2021,Un idiophone de l’Âge du Bronze en Égée : le s...,"0031-0387,2272-7639",0031-0387,Pallas,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.4000/pallas.19740,fr,journal-article


In [18]:

requests.get(CREF_API_ENDPOINT + "/doi/10.3917/cliop.028.0029").json()

{'status': 'ok',
 'message-type': 'work',
 'message-version': '1.0.0',
 'message': {'indexed': {'date-parts': [[2024, 12, 10]],
   'date-time': '2024-12-10T23:59:44Z',
   'timestamp': 1733875184744,
   'version': '3.30.1'},
  'reference-count': 23,
  'publisher': 'CAIRN',
  'issue': '2',
  'content-domain': {'domain': [], 'crossmark-restriction': False},
  'short-container-title': [],
  'abstract': '<jats:p>En un moment historique où la demande adressée aux experts en éducation et formation est de plus en plus présente sous une forme réductionniste et médicalisante, le «\xa0cas Schreber\xa0» permet de repenser les conditions de possibilité de l’éducation d’un enfant. S’il est souvent mobilisé pour montrer les méfaits d’une éducation qualifiée d’autoritaire, pour nous, il est plutôt l’occasion de montrer les risques délétères des initiatives prétendument scientifiques en éducation, structurées autour du déni de la castration, dont son seul témoignage est la condition nécessaire pour qu’

In [30]:
list_row = []
list_author = []
for n, x in enumerate(tqdm.tqdm(df1b.id, total=len(df1b))):
    doi = df1b.doi.iloc[n]
    query_response = requests.get(CREF_API_ENDPOINT + '/' + doi).json()
    try:
        abstract = query_response["message"]["abstract"].replace('<jats:p>','').replace('</jats:p>','')
    except:
        abstract = "no abstract"

    dict_row = {"id":x,
               "doi":doi,
               "abstract":abstract,
                "source":query_response["message"]['resource']['primary']['URL']
               }
    list_row.append(dict_row)
    try:
        for a in query_response["message"]["author"]:
            if len(a["affiliation"]) > 0 :
                affil = a["affiliation"][0]['name']
            else:
                affil = "no affil"
            dic_author = {'id':x,
                          'doi':doi,
                          'given_name':a["given"],
                          'family_name':a['family'],
                          'affiliation': affil
                         }
            list_author.append(dic_author)
    except:
        dic_author = {'id':x,
                          'doi':doi,
                          'given_name':'no author',
                          'family_name':'no author',
                          'affiliation': 'no affil'
                     }
        list_author.append(dic_author)
        

        
    
    


100%|██████████| 11009/11009 [57:33<00:00,  3.19it/s] 


In [36]:
dfabs = pd.DataFrame.from_dict(list_row)
dfabs.loc[dfabs.abstract!="no abstract"]

,id,doi,abstract,source
0,doi10.3917/cliop.028.0029,10.3917/cliop.028.0029,En un moment historique où la demande adressée...,https://www.cairn.info/revue-cliopsy-2022-2-pa...
1,doi10.1590/198053142861,10.1590/198053142861,"Nos últimos trinta anos, a prioridade dada a u...",http://www.scielo.br/scielo.php?script=sci_art...
3,doi10.4000/hybrid.460,10.4000/hybrid.460,"In the wake of postmodernist experiments, Chat...",https://journals.openedition.org/hybrid/460
7,doi10.4000/danse.4410,10.4000/danse.4410,Il est admis par l’historiographie que les hér...,https://journals.openedition.org/danse/4410
9,doi10.3917/arss.228.0004,10.3917/arss.228.0004,"Peu étudiés, les immigrés et descendants d’imm...",https://www.cairn.info/revue-actes-de-la-reche...
...,...,...,...,...
10999,doi10.3917/ls.173.0205,10.3917/ls.173.0205,Le contexte même des échanges électroniques (i...,https://www.cairn.info/revue-langage-et-societ...
11000,doi10.1177/0073275317751316,10.1177/0073275317751316,Using as case studies the initiatives develop...,https://journals.sagepub.com/doi/10.1177/00732...
11002,doi10.3917/cdlj.1901.0117,10.3917/cdlj.1901.0117,Résumé Nous souhaitons dans cet article expose...,https://droit.cairn.info/revue-les-cahiers-de-...
11003,doi10.3390/commodities2020010,10.3390/commodities2020010,Asset markets have long contained a section de...,https://www.mdpi.com/2813-2432/2/2/10


In [35]:
dfauthor = pd.DataFrame.from_dict(list_author)
len(dfauthor.loc[dfauthor.affiliation!="no affil"])/len(dfauthor)

0.19884835809544504

In [45]:
dfabs["lang"]= dfabs.id.map(dict(zip(df1.id, df1.lang)).get)
dfabs["type"]= dfabs.id.map(dict(zip(df1.id, df1.pub_type)).get)

In [47]:
dfabs.loc[(dfabs.abstract!="no abstract") & (dfabs.lang=="fr")]

,id,doi,abstract,source,lang,type
0,doi10.3917/cliop.028.0029,10.3917/cliop.028.0029,En un moment historique où la demande adressée...,https://www.cairn.info/revue-cliopsy-2022-2-pa...,fr,journal-article
7,doi10.4000/danse.4410,10.4000/danse.4410,Il est admis par l’historiographie que les hér...,https://journals.openedition.org/danse/4410,fr,journal-article
9,doi10.3917/arss.228.0004,10.3917/arss.228.0004,"Peu étudiés, les immigrés et descendants d’imm...",https://www.cairn.info/revue-actes-de-la-reche...,fr,NaN
12,doi10.3917/sr.040.0111,10.3917/sr.040.0111,Résumé La production critique des femmes sur l...,https://www.cairn.info/revue-societes-et-repre...,fr,NaN
15,doi10.3917/machr.244.0025,10.3917/machr.244.0025,Avant de se trouver contesté visiblement au cœ...,https://www.cairn.info/revue-maghreb-machrek-2...,fr,journal-article
...,...,...,...,...,...,...
10995,doi10.3917/anso.181.0155,10.3917/anso.181.0155,Alors que le Programme national pour la rénova...,https://www.cairn.info/revue-l-annee-sociologi...,fr,journal-article
10997,doi10.4000/echogeo.15296,10.4000/echogeo.15296,"Le 19 septembre 2017, trente-deux ans jour pou...",https://journals.openedition.org/echogeo/15296,fr,journal-article
10999,doi10.3917/ls.173.0205,10.3917/ls.173.0205,Le contexte même des échanges électroniques (i...,https://www.cairn.info/revue-langage-et-societ...,fr,journal-article
11002,doi10.3917/cdlj.1901.0117,10.3917/cdlj.1901.0117,Résumé Nous souhaitons dans cet article expose...,https://droit.cairn.info/revue-les-cahiers-de-...,fr,journal-article


In [51]:
file_path = BUCKET_OUT + "/" + "Data_bso/outputs/enriched_data/abstracts/2026-03-11_publis_with_abstract.csv"
with fs.open(file_path, "w") as file_out:
    dfabs.to_csv(file_out, sep=",", index= False)


In [52]:
file_path = BUCKET_OUT + "/" + "Data_bso/outputs/enriched_data/geolocation/2026-03-11_affil_crossref.csv"
with fs.open(file_path, "w") as file_out:
    dfauthor.to_csv(file_out, sep=",", index= False)